In [1]:
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchvision.transforms.functional as tvF
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.axes_grid1 import make_axes_locatable
import math

# ---------------------------------------------------------
# 1. Helper Functions
# ---------------------------------------------------------

def confusion_cal(y_test, py_test, y_type):
    class_num = len(np.unique(y_test))
    confusion_num = np.zeros((class_num, class_num))
    y_test = torch.tensor(y_test)
    py_test = torch.tensor(py_test)
    if y_type == 0:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(torch.argmax(py_test, dim=1) == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))
    elif y_type == 1:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(py_test == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))

    return confusion_num

def extract_handcrafted_features(spk_matrix):
    """
    Extracts Mean Firing Rate, Mean Burst Fraction, and Average ISI from a spike train matrix.
    Assumes spk_matrix has shape (num_samples, time_steps_in_ms).
    """
    num_samples = spk_matrix.shape[0]
    duration_s = spk_matrix.shape[1] / 1000.0  # Convert ms to seconds

    fr_list = np.zeros((num_samples, 1))
    bf_list = np.zeros((num_samples, 1))
    isi_list = np.zeros((num_samples, 1))

    for i in range(num_samples):
        # Find indices (times in ms) where spikes occur
        spike_times = np.where(spk_matrix[i, :] > 0)[0]
        total_spikes = len(spike_times)

        # Feature 1: Mean Firing Rate (Hz)
        fr_list[i, 0] = total_spikes / duration_s

        if total_spikes > 1:
            # Calculate Inter-Spike Intervals (in ms)
            isis = np.diff(spike_times)

            # Feature 3: Average Inter-Spike Interval
            isi_list[i, 0] = np.mean(isis)

            # Feature 2: Mean Burst Fraction
            # Based on paper: Bursts are spikes preceded by an interval < 10 ms
            burst_spikes_count = np.sum(isis < 10)
            bf_list[i, 0] = burst_spikes_count / total_spikes
        else:
            isi_list[i, 0] = 0
            bf_list[i, 0] = 0

    # Feature 4: Concatenate all three features
    combined_features = np.hstack((fr_list, bf_list, isi_list))

    return {
        'FiringRate': fr_list,
        'BurstFraction': bf_list,
        'ISI': isi_list,
        'Combined': combined_features
    }


# ---------------------------------------------------------
# 2. Data Loading (Strictly Copied)
# ---------------------------------------------------------

dat_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/data'

SPK = []
ptype = []
for rnum in range(0, 30):
    SPK.append(np.load(dat_dir + f'/FixedRepresentation/Spk_an_wn_{rnum}.npy'))
    ptype.append(np.load(dat_dir + f'/FixedRepresentation/Ptype_an_wn_{rnum}.npy'))

SPK = np.concatenate(SPK, axis=0)
ptype = np.concatenate(ptype, axis=0)

SPK1 = []
ptype1 = []
for rnum in range(0, 49):
    SPK1.append(np.load(dat_dir + f'/FixedRepresentation01/Spk_an_wn_{rnum}.npy'))
    ptype1.append(np.load(dat_dir + f'/FixedRepresentation01/Ptype_an_wn_{rnum}.npy'))

SPK1 = np.concatenate(SPK1, axis=0)
ptype1 = np.concatenate(ptype1, axis=0)

# ---------------------------------------------------------
# 3. Feature Extraction
# ---------------------------------------------------------
print("Extracting features for In-Distribution (Training/Test) Data...")
features_SPK = extract_handcrafted_features(SPK)

print("Extracting features for Out-of-Distribution Data...")
features_SPK1 = extract_handcrafted_features(SPK1)

feature_names = ['FiringRate', 'BurstFraction', 'ISI', 'Combined']

# Dictionary to hold the accuracy metrics for each feature combination
results = {
    feat: {
        'acu_set': np.zeros((50, 6)),
        'acu_set_test': np.zeros((50, 6))
    }
    for feat in feature_names
}

# ---------------------------------------------------------
# 4. Training and Test Dataset Split & Generation (Strictly Copied Logic)
# ---------------------------------------------------------
print("Starting SVM Training/Testing loop across 50 iterations...")

for ii in range(0, 50):

    # Strictly copied split generation
    train_ind = np.random.choice(np.size(SPK,axis=0), np.int32(0.8 * np.size(SPK,axis=0)), replace=False)
    test_ind = np.setdiff1d(np.arange(0, np.size(SPK,axis=0)), train_ind)
    test_ind_od = np.random.choice(np.size(SPK1,axis=0), np.int32(0.2 * np.size(SPK1,axis=0)), replace=False)

    y_train = ptype[train_ind]
    y_test = ptype[test_ind]
    y_test_od = ptype1[test_ind_od]

    # Evaluate SVM for each handcrafted feature type
    for feat in feature_names:
        x_train = features_SPK[feat][train_ind]
        x_test = features_SPK[feat][test_ind]
        x_test_od = features_SPK1[feat][test_ind_od]

        # Scaling the features is highly recommended for SVMs, especially for the "Combined"
        # dataset where features (Hz, percentages, ms) are on different scales.
        scaler = StandardScaler()
        x_train_scaled = scaler.fit_transform(x_train)
        x_test_scaled = scaler.transform(x_test)
        x_test_od_scaled = scaler.transform(x_test_od)

        # Initialize model (Added dual=False to prevent convergence warnings with scaling)
        model = svm.LinearSVC(dual=False, max_iter=10000)

        # Fit the model to training data
        model.fit(x_train_scaled, y_train)

        # Check test set accuracy
        results[feat]['acu_set'][ii, 0] = model.score(x_test_scaled, y_test)
        results[feat]['acu_set_test'][ii, 0] = model.score(x_test_od_scaled, y_test_od)

        # Class-specific accuracy
        for jj in range(0, 5):
            # In-distribution tests
            test_indices_class = np.argwhere(y_test == jj).flatten()
            if len(test_indices_class) > 0:
                results[feat]['acu_set'][ii, jj + 1] = model.score(
                    x_test_scaled[test_indices_class],
                    y_test[test_indices_class]
                )

            # Out-of-distribution tests
            od_indices_class = np.argwhere(y_test_od == jj).flatten()
            if len(od_indices_class) > 0:
                results[feat]['acu_set_test'][ii, jj + 1] = model.score(
                    x_test_od_scaled[od_indices_class],
                    y_test_od[od_indices_class]
                )

# ---------------------------------------------------------
# 5. Output / Saving Results
# ---------------------------------------------------------
save_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/result_20260601'
os.makedirs(save_dir, exist_ok=True)

for feat in feature_names:
    print(f"\n--- Results for Feature: {feat} ---")

    mean_in_dist = np.round(results[feat]['acu_set'].mean(axis=0), 3)
    mean_out_dist = np.round(results[feat]['acu_set_test'].mean(axis=0), 3)

    print("In-Distribution Accuracies:")
    print(f"  Overall: {mean_in_dist[0]:.3f}")
    for c in range(5):
        print(f"  Class {c}: {mean_in_dist[c+1]:.3f}")

    print("\nOut-of-Distribution Accuracies:")
    print(f"  Overall: {mean_out_dist[0]:.3f}")
    for c in range(5):
        print(f"  Class {c}: {mean_out_dist[c+1]:.3f}")

    # Save main arrays (Col 0: Overall, Col 1-5: Class 0-4)
    np.save(f"{save_dir}/svm_fixedrep_{feat}_in_dist.npy", results[feat]['acu_set'])
    np.save(f"{save_dir}/svm_fixedrep_{feat}_out_dist.npy", results[feat]['acu_set_test'])

    # Save individual accuracy arrays for each class
    for c in range(5):
        np.save(f"{save_dir}/svm_fixedrep_{feat}_in_dist_class_{c}.npy", results[feat]['acu_set'][:, c+1])
        np.save(f"{save_dir}/svm_fixedrep_{feat}_out_dist_class_{c}.npy", results[feat]['acu_set_test'][:, c+1])

print("\nFinished! You can now use these accuracy matrices to generate a supplementary table comparing handcrafted features vs your deep models.")

Extracting features for In-Distribution (Training/Test) Data...
Extracting features for Out-of-Distribution Data...
Starting SVM Training/Testing loop across 50 iterations...

--- Results for Feature: FiringRate ---
In-Distribution Accuracies:
  Overall: 0.487
  Class 0: 0.468
  Class 1: 0.088
  Class 2: 0.126
  Class 3: 0.800
  Class 4: 0.975

Out-of-Distribution Accuracies:
  Overall: 0.448
  Class 0: 0.286
  Class 1: 0.066
  Class 2: 0.035
  Class 3: 0.911
  Class 4: 0.946

--- Results for Feature: BurstFraction ---
In-Distribution Accuracies:
  Overall: 0.326
  Class 0: 0.035
  Class 1: 0.841
  Class 2: 0.044
  Class 3: 0.097
  Class 4: 0.621

Out-of-Distribution Accuracies:
  Overall: 0.231
  Class 0: 0.002
  Class 1: 1.000
  Class 2: 0.000
  Class 3: 0.000
  Class 4: 0.175

--- Results for Feature: ISI ---
In-Distribution Accuracies:
  Overall: 0.422
  Class 0: 0.191
  Class 1: 0.068
  Class 2: 0.135
  Class 3: 0.861
  Class 4: 0.880

Out-of-Distribution Accuracies:
  Overall: 0.